In [9]:
# Instalacja bibliotek
!pip install kagglehub --quiet
!pip install tensorflow --quiet

import kagglehub
import os
import shutil
from pathlib import Path

# Pobieranie dataset Cats vs Dogs

path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
print("Pobrano do:", path)

original_cat_dir = os.path.join(path, "PetImages", "Cat")
original_dog_dir = os.path.join(path, "PetImages", "Dog")

# Przygotowanie nowego folder z 200 kotami i 200 psami

base_dir = "/content/cats_vs_dogs_200"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

for folder in [train_dir, val_dir]:
    os.makedirs(os.path.join(folder, "Cat"), exist_ok=True)
    os.makedirs(os.path.join(folder, "Dog"), exist_ok=True)

def copy_images(src, dst, count):
    copied = 0
    for fname in os.listdir(src):
        src_path = os.path.join(src, fname)
        dst_path = os.path.join(dst, fname)
        try:
            with open(src_path, "rb") as f:
                f.read()
            shutil.copy(src_path, dst_path)
            copied += 1
            if copied >= count:
                break
        except:
            continue

# Podział 240 do treningu, 60 do walidacji
copy_images(original_cat_dir, os.path.join(train_dir, "Cat"), 160)
copy_images(original_cat_dir, os.path.join(val_dir, "Cat"), 40)

copy_images(original_dog_dir, os.path.join(train_dir, "Dog"), 160)
copy_images(original_dog_dir, os.path.join(val_dir, "Dog"), 40)

print("Dane podzielone na 160 train + 40 val dla każdej klasy")


# ImageDataGenerator - wczytanie danych
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_height, img_width = 224, 224
batch_size = 16

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = datagen.flow_from_directory(
    val_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)

Pobrano do: /kaggle/input/dog-and-cat-classification-dataset
Dane podzielone na 160 train + 40 val dla każdej klasy
Found 640 images belonging to 2 classes.
Found 160 images belonging to 2 classes.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Upewnij się, że działa na CPU
tf.config.set_visible_devices([], 'GPU')  # Wyłącza GPU

# Parametry
input_shape = (224, 224, 3)
num_classes = 1  # binarna klasyfikacja (Cat vs Dog)
epochs = 5

# Budowanie modelu ResNet50 od ZERA (bez transfer learningu)
model = tf.keras.applications.ResNet50(
    include_top=False,
    weights=None,  # brak transfer learningu
    input_shape=input_shape,
    pooling='avg'
)

# Dodanie klasyfikatora
x = model.output
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

full_model = models.Model(inputs=model.input, outputs=output)

# Kompilacja
full_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Start pomiaru czasu
start_time = time.time()

# Trenowanie
history = full_model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator
)

# Zakończenie pomiaru
end_time = time.time()
training_time = end_time - start_time

print(f"⏱Czas treningu na CPU: {training_time:.2f} sekund")


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


20/20 ━━━━━━━━━━━━━━━━━━━━ 292s 12s/step - accuracy: 0.4949 - loss: 2.1171 - val_accuracy: 0.5000 - val_loss: 0.6966
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 232s 12s/step - accuracy: 0.5447 - loss: 1.3876 - val_accuracy: 0.5000 - val_loss: 0.6986
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 222s 11s/step - accuracy: 0.4761 - loss: 1.2964 - val_accuracy: 0.5000 - val_loss: 0.6927
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 228s 11s/step - accuracy: 0.4671 - loss: 0.8745 - val_accuracy: 0.5000 - val_loss: 0.6924
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 228s 11s/step - accuracy: 0.6277 - loss: 0.7068 - val_accuracy: 0.5000 - val_loss: 0.6982
⏱Czas treningu na CPU: 1202.92 sekund


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Sprawdzenie, czy jest GPU
print("Czy dostępne GPU:", tf.config.list_physical_devices('GPU'))

# Parametry
input_shape = (224, 224, 3)
epochs = 5

# Budowanie modelu ResNet50 od zera (bez transfer learningu)
model = tf.keras.applications.ResNet50(
    include_top=False,
    weights=None,
    input_shape=input_shape,
    pooling='avg'
)

x = model.output
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)
full_model = models.Model(inputs=model.input, outputs=output)

full_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Trening z GPU
start_gpu = time.time()
history_gpu = full_model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator
)
end_gpu = time.time()

print(f"⏱ Czas treningu na GPU: {end_gpu - start_gpu:.2f} sekund")


Czy dostępne GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


20/20 ━━━━━━━━━━━━━━━━━━━━ 70s 389ms/step - accuracy: 0.5171 - loss: 1.8486 - val_accuracy: 0.5000 - val_loss: 0.6936
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - accuracy: 0.4676 - loss: 1.3786 - val_accuracy: 0.5000 - val_loss: 0.7048
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 195ms/step - accuracy: 0.5288 - loss: 1.0401 - val_accuracy: 0.5000 - val_loss: 0.6900
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 175ms/step - accuracy: 0.5394 - loss: 0.8562 - val_accuracy: 0.5000 - val_loss: 0.7770
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 176ms/step - accuracy: 0.6007 - loss: 0.6497 - val_accuracy: 0.5000 - val_loss: 0.7249
⏱ Czas treningu na GPU: 87.59 sekund


## Porównanie treningu ResNet50 od zera (5 epok) na CPU i GPU

### Parametry eksperymentu
- Liczba próbek: 400 (200 na klasę)
- Rozmiar obrazów: 224x224
- Liczba epok: 5
- Model: ResNet50 od zera (bez transfer learningu), z klasyfikatorem binarnym

---

### CPU (baseline)
- Czas treningu: 1202 s  
- Dokładność treningowa: 62.77%  
- Dokładność walidacyjna: 50%  

---

### GPU
- Czy dostępne GPU: `[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]`  
- Czas treningu: 87.59 s  
- Dokładność treningowa: 60.07%  
- Dokładność walidacyjna: 50%  

###Wnioski
Przyspieszenie treningu na GPU jest bardzo wyraźne – ponad 13x szybciej niż na CPU.
Dokładność walidacyjna pozostaje na niskim poziomie (50%), co może sugerować, że
potrzebna będzie optymalizacja dokładności (normalizacja, augmentacja, więcej danych lub epok).

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Parametry
input_shape = (224, 224, 3)
epochs = 10

# Transfer learning - ResNet50 z wstępnie wytrenowanymi wagami (ImageNet)
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape,
    pooling='avg'
)

# Zamrażamy warstwy bazowe na początku
base_model.trainable = False

x = base_model.output
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model_tl = models.Model(inputs=base_model.input, outputs=output)

model_tl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Trening z zamrożonymi wagami
start_tl = time.time()
history_tl = model_tl.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator
)
end_tl = time.time()
training_time_tl = end_tl - start_tl
print(f"⏱ Czas treningu transfer learning (zamrożone warstwy): {training_time_tl:.2f} sekund")

# Odmrażamy ostatnie warstwy i trenujemy dalej (fine-tuning)
base_model.trainable = True

# Kompilujemy ponownie z niższym learning rate
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

start_ft = time.time()
history_ft = model_tl.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator
)
end_ft = time.time()
training_time_ft = end_ft - start_ft
print(f"⏱ Czas treningu fine-tuning: {training_time_ft:.2f} sekund")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 16s 252ms/step - accuracy: 0.5163 - loss: 0.8299 - val_accuracy: 0.5125 - val_loss: 0.7032
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step - accuracy: 0.5602 - loss: 0.7589 - val_accuracy: 0.5250 - val_loss: 0.6937
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.4895 - loss: 0.7664 - val_accuracy: 0.6500 - val_loss: 0.6532
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.5317 - loss: 0.6851 - val_accuracy: 0.6375 - val_loss: 0.6546
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.5240 - loss: 0.6803 - val_accuracy: 0.7375 - val_loss: 0.6505
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.6077 - loss: 0.6608 - val_accuracy: 0.6250 - val_loss: 0.6523
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.6341 - loss: 0.6577 - val_accuracy: 0.6125 - val_loss: 0.6506
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accur

1. Transfer learning (zamrożone warstwy)
- Czas treningu: 28.21 s
- Dokładność treningowa (max): 64.05%
- Dokładność walidacyjna (max): 73.75%
2. Fine-tuning (odmrożenie warstw)
- Czas treningu: ok. 108 s
- Dokładność treningowa (max): 100%
- Dokładność walidacyjna (max): 58.75% - może sugerować overfitting

Wniosek: zamrożenie warstw podczas transfer learningu jest korzystniejsze w kontekście uzyskania lepszej jakości na zbiorze walidacyjnym i szybszego treningu, unikając nadmiernego dopasowania.


In [2]:

# Generator z normalizacją
datagen_norm = ImageDataGenerator(rescale=1./255)

# Generator bez normalizacji
datagen_raw = ImageDataGenerator()  # bez przeskalowania

# Generatory danych
train_generator_norm = datagen_norm.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

val_generator_norm = datagen_norm.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

train_generator_raw = datagen_raw.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

val_generator_raw = datagen_raw.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)


Found 320 images belonging to 2 classes.
Found 80 images belonging to 2 classes.
Found 320 images belonging to 2 classes.
Found 80 images belonging to 2 classes.


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Parametry
input_shape = (224, 224, 3)
epochs = 5  # na potrzeby porównania

# Transfer learning - ResNet50 z wstępnie wytrenowanymi wagami (ImageNet)
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape,
    pooling='avg'
)

# Zamrażamy warstwy bazowe na początku
base_model.trainable = False

x = base_model.output
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model_tl = models.Model(inputs=base_model.input, outputs=output)

model_tl.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Trening z normalizacja

# Klonowanie modelu
model_raw = tf.keras.models.clone_model(model_tl)
model_raw.set_weights(model_tl.get_weights())  # kopia wag z model_tl

model_raw.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

start_raw = time.time()
history_raw = model_raw.fit(
    train_generator_raw,  # generator BEZ normalizacji
    epochs=epochs,
    validation_data=val_generator_raw
)
end_raw = time.time()
print(f"⏱ Czas treningu bez normalizacji: {end_raw - start_raw:.2f} sekund")

# trening bez normalizacji

start_norm = time.time()
history_norm = model_tl.fit(
    train_generator_norm,  # generator Z normalizacją
    epochs=epochs,
    validation_data=val_generator_norm
)
end_norm = time.time()
print(f"⏱ Czas treningu z normalizacją: {end_norm - start_norm:.2f} sekund")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 338ms/step - accuracy: 0.8006 - loss: 0.4314 - val_accuracy: 0.9625 - val_loss: 0.0772
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.9438 - loss: 0.1500 - val_accuracy: 0.9750 - val_loss: 0.0648
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.9791 - loss: 0.0490 - val_accuracy: 1.0000 - val_loss: 0.0035
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.9860 - loss: 0.0724 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 1.0000 - loss: 0.0049 - val_accuracy: 1.0000 - val_loss: 2.9826e-04
⏱ Czas treningu bez normalizacji: 34.94 sekund
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 278ms/step - accuracy: 0.4213 - loss: 0.9254 - val_accuracy: 0.5750 - val_loss: 0.6766
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.5088 - loss: 0.7295 - val_accuracy: 0.5500 - val_loss: 0.6634
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0

# Raport Normalizacja danych a trening modelu ResNet50

**Model:** ResNet50 (transfer learning, warstwy zamrożone)  
**Dane:** Cats vs Dogs 160 trening 40 walidacja  5 epok, batch=16

Wyniki treningu bez normalizacji:  
- Dokładność treningowa: 100%  
- Dokładność walidacyjna: 100%  
- Czas treningu: 34.94 sekund

Wyniki treningu z normalizacją:  
- Dokładność treningowa: 60.05%  
- Dokładność walidacyjna: 61.25%  
- Czas treningu: 24.05 sekund

**Wnioski:**  
Model bez normalizacji osiągnął lepsze wyniki niż ten z normalizacją, co jest nietypowe. Normalizacja danych zazwyczaj poprawia efektywność uczenia.


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Parametry
img_height, img_width = 224, 224
batch_size = 16

# Generator BEZ augmentacji (tylko normalizacja)
datagen_norm = ImageDataGenerator(rescale=1./255)

train_generator_norm = datagen_norm.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)

val_generator_norm = datagen_norm.flow_from_directory(
    val_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)

# Generator Z augmentacją
datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,          # losowe obracanie obrazów do 20 stopni
    width_shift_range=0.1,      # losowe przesunięcie poziome do 10% szerokości
    height_shift_range=0.1,     # losowe przesunięcie pionowe do 10% wysokości
    zoom_range=0.1,             # losowe powiększenie/zmniejszenie obrazu do 10%
    horizontal_flip=True,       # losowe odbicie lustrzane w poziomie
    fill_mode='nearest'         # sposób wypełniania pikseli po transformacji
)

train_generator_aug = datagen_aug.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)


Found 320 images belonging to 2 classes.
Found 80 images belonging to 2 classes.
Found 320 images belonging to 2 classes.


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

input_shape = (img_height, img_width, 3)
epochs = 5

# Model (ResNet50 + dense)
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape,
    pooling='avg'
)
base_model.trainable = False

x = base_model.output
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model_base = models.Model(inputs=base_model.input, outputs=output)
model_base.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Model bez augmentacji (z normalizacją)
model_norm = tf.keras.models.clone_model(model_base)
model_norm.set_weights(model_base.get_weights())
model_norm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

start_norm = time.time()
history_norm = model_norm.fit(
    train_generator_norm,
    epochs=epochs,
    validation_data=val_generator_norm
)
end_norm = time.time()

# Model z augmentacją
model_aug = tf.keras.models.clone_model(model_base)
model_aug.set_weights(model_base.get_weights())
model_aug.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

start_aug = time.time()
history_aug = model_aug.fit(
    train_generator_aug,
    epochs=epochs,
    validation_data=val_generator_norm
)
end_aug = time.time()


Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 313ms/step - accuracy: 0.4695 - loss: 0.8766 - val_accuracy: 0.5875 - val_loss: 0.6617
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - accuracy: 0.4696 - loss: 0.7669 - val_accuracy: 0.5000 - val_loss: 0.7203
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.4722 - loss: 0.7923 - val_accuracy: 0.6000 - val_loss: 0.6656
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.5333 - loss: 0.7072 - val_accuracy: 0.5750 - val_loss: 0.6900
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 70ms/step - accuracy: 0.6021 - loss: 0.6973 - val_accuracy: 0.5500 - val_loss: 0.6629
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 462ms/step - accuracy: 0.5367 - loss: 0.9024 - val_accuracy: 0.5625 - val_loss: 0.6756
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 276ms/step - accuracy: 0.5204 - loss: 0.7576 - val_accuracy: 0.5000 - val_loss: 0.7731
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 239ms/step - accuracy: 0.5402 - loss: 0.7533 - val_accuracy: 0.6000 - val

#Porównanie wyników: z augmentacją vs bez augmentacji

Bez augmentacji:
- Dokładność treningowa: ~47–60%
- Dokładność walidacji: ~50–66%
Z augmentacją:
Dokładność treningowa: podobna, ~54–55%
Dokładność walidacji: lekko wyższa, do ~66%
Wyniki nieco stabilniejsze, ale nadal niskie
Augmentacja nieznacznie pomaga, ale efekt jest niewielki

Wnioski:
Model wymaga dalszej optymalizacji — np. innej architektury, większej liczby danych lub dłuższego treningu

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

input_shape = (224, 224, 3)
epochs = 5
batch_size = 16

# Transfer learning - baza ResNet50
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape,
    pooling='avg'
)
base_model.trainable = False  # zamrażamy warstwy bazowe

#Model z dropout
x = base_model.output
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)  # dropout 50%
output = layers.Dense(1, activation='sigmoid')(x)
model_dropout = models.Model(inputs=base_model.input, outputs=output)

model_dropout.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#Model bez dropout
x2 = base_model.output
x2 = layers.Dense(256, activation='relu')(x2)
output2 = layers.Dense(1, activation='sigmoid')(x2)
model_no_dropout = models.Model(inputs=base_model.input, outputs=output2)

model_no_dropout.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#Trening modelu z dropout

start_dropout = time.time()
history_dropout = model_dropout.fit(
    train_generator_norm,
    epochs=epochs,
    validation_data=val_generator_norm
)
end_dropout = time.time()
print(f"Czas treningu z dropout: {end_dropout - start_dropout:.2f} s")

#Trening modelu bez dropout
print("Trening modelu bez dropout...")
start_no_dropout = time.time()
history_no_dropout = model_no_dropout.fit(
    train_generator_norm,
    epochs=epochs,
    validation_data=val_generator_norm
)
end_no_dropout = time.time()
print(f"Czas treningu bez dropout: {end_no_dropout - start_no_dropout:.2f} s")


Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 255ms/step - accuracy: 0.5431 - loss: 0.8262 - val_accuracy: 0.5000 - val_loss: 0.7685
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.5687 - loss: 0.7204 - val_accuracy: 0.6250 - val_loss: 0.6626
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.5037 - loss: 0.7386 - val_accuracy: 0.5000 - val_loss: 0.7601
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - accuracy: 0.4943 - loss: 0.7605 - val_accuracy: 0.5000 - val_loss: 0.6987
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - accuracy: 0.4938 - loss: 0.7300 - val_accuracy: 0.6000 - val_loss: 0.6568
Czas treningu z dropout: 21.11 s
Trening modelu bez dropout...
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 271ms/step - accuracy: 0.4792 - loss: 0.8638 - val_accuracy: 0.5625 - val_loss: 0.6630
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.5549 - loss: 0.6729 - val_accuracy: 0.5875 - val_loss: 0.6511
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step -

Model z dropout:

- Dokładność treningowa: od 49% do 57%
- Dokładność walidacji: od 50% do 62.5%
- Czas treningu: ~21.1 s

Model bez dropout:

- Dokładność treningowa: od 48% do 60.4%
- Dokładność walidacji: utrzymuje się w okolicach 56-61%
- Czas treningu: ~21.7 s

Wnioski:
Model bez dropout osiąga wyższą dokładność i niższą stratę zarówno na treningu, jak i walidacji. Dropout w tym przypadku nie poprawił skuteczności, a wręcz może utrudniać naukę modelu, szczególnie w tak krótkim treningu (5 epok).

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import time
import matplotlib.pyplot as plt

# Parametry
input_shape = (224, 224, 3)
batch_size = 64
epochs = 5

# Przetwarzanie danych
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

def get_data_generators(data_path):
    train_gen = datagen.flow_from_directory(
        data_path,
        target_size=input_shape[:2],
        batch_size=batch_size,
        class_mode='binary',
        subset='training'
    )
    val_gen = datagen.flow_from_directory(
        data_path,
        target_size=input_shape[:2],
        batch_size=batch_size,
        class_mode='binary',
        subset='validation'
    )
    return train_gen, val_gen

# A. Dane podstawowe
train_gen_base, val_gen_base = get_data_generators("/content/cats_vs_dogs_200")

# B. Dane z dołożeniem zdjęć
train_gen_extended, val_gen_extended = get_data_generators("/content/cats_vs_dogs_400")

def build_model():
    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        pooling='avg'
    )
    base_model.trainable = False

    x = base_model.output
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

#Trenowanie

def train_model(name, model, train_gen, val_gen):
    print(f"\nTrening: {name}")
    start = time.time()
    history = model.fit(train_gen, epochs=epochs, validation_data=val_gen)
    end = time.time()
    duration = end - start
    print(f"⏱ Czas treningu ({name}): {duration:.2f} s")
    return history, duration

# Model bez dodatkowych danych
model_base = build_model()
history_base, time_base = train_model("Bez dodatkowych danych", model_base, train_gen_base, val_gen_base)

# Model z dodatkowymi danymi
model_extended = build_model()
history_ext, time_ext = train_model("Z dodatkowymi danymi", model_extended, train_gen_extended, val_gen_extended)



Found 320 images belonging to 2 classes.
Found 80 images belonging to 2 classes.
Found 640 images belonging to 2 classes.
Found 160 images belonging to 2 classes.

Trening: Bez dodatkowych danych
Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.7165 - loss: 0.6790 - val_accuracy: 0.8000 - val_loss: 0.5154
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 273ms/step - accuracy: 0.7323 - loss: 0.5985 - val_accuracy: 0.8000 - val_loss: 0.4988
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 272ms/step - accuracy: 0.7971 - loss: 0.5870 - val_accuracy: 0.8000 - val_loss: 0.5809
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 272ms/step - accuracy: 0.7902 - loss: 0.6172 - val_accuracy: 0.8000 - val_loss: 0.5171
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 270ms/step - accuracy: 0.7498 - loss: 0.5848 - val_accuracy: 0.8000 - val_loss: 0.4985
⏱ Czas treningu (Bez dodatkowych danych): 24.76 s

Trening: Z dodatkowymi danymi
Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 18s 884ms/step - accuracy: 0.7121 - loss: 0.6450 - val_accuracy

# Porównanie treningu bez dodatkowych danych i z dodatkowymi danymi

- **Liczba obrazów treningowych:**
  - Bez dodatkowych danych: 320
  - Z dodatkowymi danymi: 640

- **Liczba obrazów walidacyjnych:**
  - Bez dodatkowych danych: 80
  - Z dodatkowymi danymi: 160

- **Dokładność na treningu:**
  - Bez dodatkowych danych: od 71.65% (epoka 1) do 74.98% (epoka 5)
  - Z dodatkowymi danymi: od 71.21% (epoka 1) do 82.27% (epoka 5)

- **Dokładność na walidacji:**
  - Stała na poziomie około 80% w obu przypadkach

- **Czas treningu:**
  - Bez dodatkowych danych: 24.76 s
  - Z dodatkowymi danymi: 28.99 s

## Wnioski:
Dodanie nowych danych zwiększyło dokładność modelu na zbiorze treningowym i nie wpłynęło negatywnie na dokładność walidacji. Trening trwał nieco dłużej, ale zyskał lepsze dopasowanie do danych.


In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Funkcja do tworzenia modelu transfer learning (ResNet50)
def create_model(input_shape):
    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        pooling='avg'
    )
    base_model.trainable = False

    x = base_model.output
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Ścieżki do danych
train_dir = "/content/cats_vs_dogs_200/train"
val_dir = "/content/cats_vs_dogs_200/val"

# Parametry treningu
batch_size = 32
epochs = 5

# Lista rozmiarów do testowania
input_sizes = [(96, 96), (160, 160), (224, 224)]

results = {}

for size in input_sizes:
    print(f"\n--- Trening dla rozmiaru wejściowego: {size} ---")

    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        train_dir,
        image_size=size,
        batch_size=batch_size,
        label_mode='binary'
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        val_dir,
        image_size=size,
        batch_size=batch_size,
        label_mode='binary'
    )

    model = create_model(input_shape=(size[0], size[1], 3))

    start_time = time.time()
    history = model.fit(
        train_ds,
        epochs=epochs,
        validation_data=val_ds,
        verbose=2
    )
    end_time = time.time()

    results[size] = {
        'train_accuracy': history.history['accuracy'][-1],
        'val_accuracy': history.history['val_accuracy'][-1],
        'training_time': end_time - start_time
    }

# Podsumowanie wyników
for size, res in results.items():
    print(f"Rozmiar {size}:")
    print(f"  Dokładność treningowa: {res['train_accuracy']:.4f}")
    print(f"  Dokładność walidacyjna: {res['val_accuracy']:.4f}")
    print(f"  Czas treningu: {res['training_time']:.2f} sekund")



--- Trening dla rozmiaru wejściowego: (96, 96) ---
Found 320 files belonging to 2 classes.
Found 80 files belonging to 2 classes.
Epoch 1/5
10/10 - 18s - 2s/step - accuracy: 0.7563 - loss: 0.8336 - val_accuracy: 0.8875 - val_loss: 0.4230
Epoch 2/5
10/10 - 9s - 903ms/step - accuracy: 0.8844 - loss: 0.3952 - val_accuracy: 0.9375 - val_loss: 0.0930
Epoch 3/5
10/10 - 0s - 49ms/step - accuracy: 0.9375 - loss: 0.2144 - val_accuracy: 0.9750 - val_loss: 0.0666
Epoch 4/5
10/10 - 1s - 67ms/step - accuracy: 0.9531 - loss: 0.1101 - val_accuracy: 0.9875 - val_loss: 0.0346
Epoch 5/5
10/10 - 0s - 44ms/step - accuracy: 0.9656 - loss: 0.1202 - val_accuracy: 1.0000 - val_loss: 0.0105

--- Trening dla rozmiaru wejściowego: (160, 160) ---
Found 320 files belonging to 2 classes.
Found 80 files belonging to 2 classes.
Epoch 1/5
10/10 - 20s - 2s/step - accuracy: 0.8219 - loss: 0.4877 - val_accuracy: 0.9000 - val_loss: 0.2442
Epoch 2/5
10/10 - 1s - 64ms/step - accuracy: 0.9406 - loss: 0.1662 - val_accuracy: 

Wniosek:

Większe rozmiary wejściowe poprawiają dokładność treningową i czas treningu jest krótszy . Dokładność walidacyjna osiąga maksymalny poziom 1.0 dla wszystkich rozmiarów, co sugeruje, że model dobrze generalizuje niezależnie od rozmiaru.

In [14]:
import tensorflow as tf
from tensorflow.keras import layers, models
import time

# Ścieżki do danych
train_dir = '/content/cats_vs_dogs_200/train'
val_dir = '/content/cats_vs_dogs_200/val'

# Parametry ogólne
input_shape = (224, 224)
epochs = 5

# Funkcja budująca model
def build_model():
    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(*input_shape, 3),
        pooling='avg'
    )
    base_model.trainable = False
    x = base_model.output
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    model = models.Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

batch_sizes = [32, 64, 128]
results = {}

for batch_size in batch_sizes:
    print(f"\n--- Trening dla batch size: {batch_size} ---")

    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        train_dir,
        image_size=input_shape,
        batch_size=batch_size,
        label_mode='binary'
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        val_dir,
        image_size=input_shape,
        batch_size=batch_size,
        label_mode='binary'
    )

    model = build_model()

    start_time = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=2
    )
    end_time = time.time()

    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    training_time = end_time - start_time

    results[batch_size] = {
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "training_time": training_time
    }



--- Trening dla batch size: 32 ---
Found 320 files belonging to 2 classes.
Found 80 files belonging to 2 classes.
Epoch 1/5
10/10 - 16s - 2s/step - accuracy: 0.8906 - loss: 0.2710 - val_accuracy: 0.9625 - val_loss: 0.1272
Epoch 2/5
10/10 - 1s - 112ms/step - accuracy: 0.9438 - loss: 0.1908 - val_accuracy: 0.9875 - val_loss: 0.0146
Epoch 3/5
10/10 - 1s - 120ms/step - accuracy: 0.9906 - loss: 0.0228 - val_accuracy: 1.0000 - val_loss: 0.0107
Epoch 4/5
10/10 - 1s - 126ms/step - accuracy: 0.9875 - loss: 0.0314 - val_accuracy: 1.0000 - val_loss: 0.0042
Epoch 5/5
10/10 - 1s - 126ms/step - accuracy: 0.9937 - loss: 0.0118 - val_accuracy: 1.0000 - val_loss: 0.0052

--- Trening dla batch size: 64 ---
Found 320 files belonging to 2 classes.
Found 80 files belonging to 2 classes.
Epoch 1/5
5/5 - 17s - 3s/step - accuracy: 0.7844 - loss: 0.4710 - val_accuracy: 0.9750 - val_loss: 0.0600
Epoch 2/5
5/5 - 1s - 220ms/step - accuracy: 0.9563 - loss: 0.1328 - val_accuracy: 0.9875 - val_loss: 0.0281
Epoch 3/

Wnioski z treningu dla różnych batch size
Batch size = 32
- Najwyższa dokładność treningowa (do 99.37%) i bardzo szybka zbieżność.
- Walidacja idealna na ostatnich epokach (100%).
- Czas na epokę jest umiarkowany, ale model dobrze się uczy i dokładność jest najwyższa.

Batch size = 64
- Nieco niższa dokładność początkowa niż dla 32, ale model szybko się poprawia i osiąga blisko 98-100% na walidacji.
- Czas na epokę jest krótszy niż przy batch 32.
- Model jest stabilny i ma bardzo dobrą jakość, choć minimalnie niższą niż batch 32.

Batch size = 128
- Najniższa dokładność początkowa (64.69%).
- Pomimo tego końcowa dokładność walidacji dochodzi do 100%, ale trening jest mniej stabilny i wymaga więcej czasu na pojedynczą iterację.
- Mniej batchy na epokę, więc mniej aktualizacji modelu, co może spowalniać naukę i powodować gorszą jakość na początku.

In [15]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet101, InceptionV3, MobileNet
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.mobilenet import preprocess_input as mobilenet_preprocess
import time

# Funkcja do przygotowania datasetu z odpowiednim preprocessingiem
def prepare_dataset(data_dir, image_size, batch_size, preprocess_func):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        image_size=image_size,
        batch_size=batch_size,
        label_mode='binary',
        shuffle=True
    )
    # Zastosuj preprocessing specyficzny dla sieci
    ds = ds.map(lambda x, y: (preprocess_func(x), y))
    return ds

# Funkcja do stworzenia modelu na bazie pretrenowanej sieci
def create_model(base_model_class, input_shape, preprocess_func):
    base_model = base_model_class(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Zamrożenie warstw bazowych

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Parametry testu
data_dir_train = "/content/cats_vs_dogs_200/train"
data_dir_val = "/content/cats_vs_dogs_200/val"
image_size = (160, 160)
batch_size = 32
epochs = 5

# Lista modeli do testu: (nazwa, klasa modelu, funkcja preprocessingowa)
models_to_test = [
    ("VGG16", VGG16, vgg_preprocess),
    ("ResNet101", ResNet101, resnet_preprocess),
    ("InceptionV3", InceptionV3, inception_preprocess),
    ("MobileNet", MobileNet, mobilenet_preprocess)
]

results = []

for name, base_class, preprocess_func in models_to_test:
    print(f"\n--- Trening modelu: {name} ---")

    train_ds = prepare_dataset(data_dir_train, image_size, batch_size, preprocess_func)
    val_ds = prepare_dataset(data_dir_val, image_size, batch_size, preprocess_func)

    model = create_model(base_class, input_shape=(*image_size, 3), preprocess_func=preprocess_func)

    start_time = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=2
    )
    elapsed_time = time.time() - start_time

    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]

    print(f"Model: {name} - Dokładność treningowa: {train_acc:.4f}, Dokładność walidacyjna: {val_acc:.4f}, Czas treningu: {elapsed_time:.2f} s")

    results.append((name, train_acc, val_acc, elapsed_time))



--- Trening modelu: VGG16 ---
Found 320 files belonging to 2 classes.
Found 80 files belonging to 2 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
10/10 - 15s - 2s/step - accuracy: 0.6500 - loss: 2.2850 - val_accuracy: 0.7500 - val_loss: 1.5770
Epoch 2/5
10/10 - 8s - 799ms/step - accuracy: 0.7656 - loss: 1.2270 - val_accuracy: 0.8250 - val_loss: 0.8967
Epoch 3/5
10/10 - 2s - 220ms/step - accuracy: 0.8313 - loss: 0.7335 - val_accuracy: 0.9000 - val_loss: 0.5865
Epoch 4/5
10/10 - 2s - 182ms/step - accuracy: 0.8906 - loss: 0.4547 - val_accuracy: 0.8875 - val_loss: 0.4016
Epoch 5/5
10/10 - 2s - 162ms/step - accuracy: 0.9094 - loss: 0.3306 - val_accuracy: 0.9375 - val_loss: 0.2851
Model: VGG16 - Dokładność treningowa: 0.9094, Dokładność walidacyjna: 0.9375, Czas treningu: 29.98 s

--- Trening modelu: ResNet101 ---
Found 320 files belonging to 2 classes.
Found 80 files belonging to 2 classes.
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
10/10 - 32s -

| Model           | Dokładność treningowa | Dokładność walidacyjna | Czas treningu (s) | Uwagi                                                                                |
| --------------- | --------------------- | ---------------------- | ----------------- | ------------------------------------------------------------------------------------ |
| **VGG16**       | 90.94%                | 93.75%                 | 29.98             | Klasyczna architektura, stabilna, ale wolniejsza.                                    |
| **ResNet101**   | 99.37%                | 100.00%                | 40.79             | Bardzo dokładny, ale najwolniejszy. Bardzo głęboka sieć.                             |
| **InceptionV3** | 99.37%                | 100.00%                | 27.81             | Wysoka precyzja, krótszy czas niż ResNet. Bardziej zrównoważony.                     |
| **MobileNet**   | 95.31%                | 97.50%                 | 14.79             | Najszybszy model, bardzo dobry kompromis jakość/czas. Idealny na urządzenia mobilne. |

Wnioski:
- najlepsza dokładność: ResNet101 i InceptionV3 osiągnęły 100% dokładności walidacyjnej, co świadczy o ich skuteczności.
- Najlepszy czas vs dokładność: MobileNet jest najszybszy (14.79 s) i mimo mniejszej złożoności osiągnął bardzo wysoką dokładność (97.50%).
- VGG16: uzyskał dobre wyniki, ale wymagał więcej czasu i osiągał niższą dokładność niż nowsze architektury.
